In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pylab as plt
from ai_edge_litert.interpreter import Interpreter
from pathlib import Path


model_dir = Path("./models")
tflite_model_file = model_dir/'model.tflite'   

interpreter = Interpreter(model_path=str(tflite_model_file))
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

input_index = input_details[0]["index"]
output_index = output_details[0]["index"]

input_shape = input_details[0]["shape"]
input_dtype = input_details[0]["dtype"]

print("Input shape expected by TFLite:", input_shape)
print("Input dtype expected by TFLite:", input_dtype)
print("Output shape:", output_details[0]["shape"])


In [ ]:
import serial
from serial.tools import list_ports
import time
from datetime import datetime
import pandas as pd

ports = list(list_ports.comports())

if not ports:
    print("No serial ports found. Check the USB cable, board connection, and drivers.")
else:
    for p in ports:
        print(f"{p.device:20s} | {p.description} | {p.hwid}")

PORT = [p for p in ports if p.description == "Nano 33 BLE"][0].device
BAUD_RATE = 9600
READ_TIME = 10
READ_TIMEOUT_SECONDS = 1

print(f"Using port: {PORT}")

points = []
print(f"Opening {PORT} at {BAUD_RATE} baud for {READ_TIME} seconds...")

with serial.Serial(PORT, BAUD_RATE, timeout=READ_TIMEOUT_SECONDS) as ser:
    time.sleep(2)
    ser.reset_input_buffer()

    print("Collecting data. Press the stop button in Jupyter to interrupt.")
    start = time.time()

    while time.time() - start < READ_TIME:
        raw = ser.readline()
        if not raw:
            continue

        line = raw.decode("utf-8", errors="replace").strip()
        if not line:
            continue
        numbers = [float(x) for x in line.split("|")]
        if len(numbers) < 2:
            print(f"Unexpected data format: '{line}'")
            continue
        input_np = np.array(numbers[1], dtype=input_dtype).reshape(input_shape)
        interpreter.set_tensor(input_index, input_np)
        interpreter.invoke()
        pred = 1 if interpreter.get_tensor(output_index)[0][0] > 0.5 else 0
        
        data_point = {"timestamp": datetime.now().isoformat(timespec="milliseconds"),
                      "raw": line,
                      "accel_x": numbers[0],   
                      "avg_accel_x": numbers[1],
                      "elapsed_seconds": time.time() - start,
                      "prediction": pred
        }
        points.append(data_point)
    df = pd.DataFrame(points)
    print(f"Collected {len(df)} data points.")

df.head(100)

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import clear_output, display

columns=("accel_x", "avg_accel_x", "prediction")
plt.figure(figsize=(10, 5))
for col in columns:
    plt.plot(df["elapsed_seconds"], df[col], label=col)
plt.xlabel("Elapsed time (seconds)")
plt.ylabel("Sensor value")
plt.title("Live Nano 33 BLE Sensor Stream")
plt.legend(loc="upper right")
plt.grid(True)
plt.show()